In [35]:
pip install pandas



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

# -----------------------------------------
# CREATE OUTPUT FOLDER
# -----------------------------------------
os.makedirs("charts", exist_ok=True)

# -----------------------------------------
# LOAD CLEANED DATA
# -----------------------------------------
df = pd.read_csv("cleaned_data/nc_2019_2024_cleaned.csv", low_memory=False)

# -----------------------------------------
# PREP
# -----------------------------------------
RACES_OF_INTEREST = [
    "White",
    "Black or African American",
    "Asian",
    "American Indian or Alaska Native"
]
df = df[df["derived_race"].isin(RACES_OF_INTEREST)].copy()

# Clean DTI — extract lower bound from ranges like "20%-<30%"
df["dti_numeric"] = (
    df["debt_to_income_ratio"]
    .astype(str)
    .str.extract(r"(\d+)")[0]
    .astype(float)
)

# Clean income (reported in thousands in HMDA)
df["income"] = pd.to_numeric(df["income"], errors="coerce")
df["income_annual"] = df["income"] * 1000

# Drop rows missing key fields
df = df.dropna(subset=["interest_rate", "dti_numeric", "income_annual", "derived_race"])

# Filter to reasonable ranges to remove outliers
df = df[
    (df["interest_rate"] > 1) & (df["interest_rate"] < 15) &
    (df["dti_numeric"] >= 0) & (df["dti_numeric"] <= 65) &
    (df["income_annual"] > 0) & (df["income_annual"] < 500000)
]

# -----------------------------------------
# BIN DTI AND INCOME
# -----------------------------------------
dti_bins   = [0, 20, 30, 40, 50, 65]
dti_labels = ["<20%", "20-30%", "30-40%", "40-50%", "50-65%"]

income_bins   = [0, 50000, 75000, 100000, 150000, 500000]
income_labels = ["<$50k", "$50-75k", "$75-100k", "$100-150k", "$150k+"]

df["dti_bin"]    = pd.cut(df["dti_numeric"],   bins=dti_bins,    labels=dti_labels,    right=False)
df["income_bin"] = pd.cut(df["income_annual"], bins=income_bins, labels=income_labels, right=False)

race_short = {
    "White": "White",
    "Black or African American": "Black",
    "Asian": "Asian",
    "American Indian or Alaska Native": "Native"
}
df["race_short"] = df["derived_race"].map(race_short)

# -----------------------------------------
# COLOR PALETTE
# -----------------------------------------
COLORS = {
    "White":  "#2C5F8A",
    "Black":  "#D95F02",
    "Asian":  "#1B9E77",
    "Native": "#7570B3"
}

races   = list(race_short.values())
n       = len(races)
width   = 0.18
offsets = np.linspace(-(n-1)/2, (n-1)/2, n) * width

# ===========================================
# CHART 1 — By DTI Bin
# ===========================================
dti_grouped = (
    df.groupby(["dti_bin", "race_short"], observed=True)["interest_rate"]
    .mean()
    .reset_index()
)

x = np.arange(len(dti_labels))

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor("#F7F9FB")
ax.set_facecolor("#F7F9FB")

for i, race in enumerate(races):
    vals = []
    for label in dti_labels:
        row = dti_grouped[(dti_grouped["dti_bin"] == label) & (dti_grouped["race_short"] == race)]
        vals.append(row["interest_rate"].values[0] if not row.empty else np.nan)
    ax.bar(x + offsets[i], vals, width=width, label=race,
           color=COLORS[race], edgecolor="white", linewidth=0.5, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(dti_labels, fontsize=11)
ax.set_xlabel("Debt-to-Income Ratio Bracket", fontsize=12, labelpad=10)
ax.set_ylabel("Average Interest Rate (%)", fontsize=12, labelpad=10)
ax.set_title(
    "Average Interest Rate by Race — Controlled for Debt-to-Income Ratio\nNorth Carolina HMDA Data, 2019–2024",
    fontsize=14, fontweight="bold", pad=15
)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f%%"))
ax.legend(title="Race", fontsize=10, title_fontsize=10, framealpha=0.8)
ax.grid(axis="y", linestyle="--", alpha=0.5, zorder=0)
ax.spines[["top", "right"]].set_visible(False)
ax.annotate(
    "Within each DTI bracket, persistent gaps between race groups\nsuggest pricing differences beyond risk-based explanations.",
    xy=(0.02, 0.97), xycoords="axes fraction", fontsize=9, color="#555555", va="top",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="#CCCCCC", alpha=0.8)
)
plt.tight_layout()
plt.savefig("charts/chart1_rate_by_dti_and_race.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: charts/chart1_rate_by_dti_and_race.png")

# ===========================================
# CHART 2 — By Income Bin
# ===========================================
income_grouped = (
    df.groupby(["income_bin", "race_short"], observed=True)["interest_rate"]
    .mean()
    .reset_index()
)

x = np.arange(len(income_labels))

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor("#F7F9FB")
ax.set_facecolor("#F7F9FB")

for i, race in enumerate(races):
    vals = []
    for label in income_labels:
        row = income_grouped[(income_grouped["income_bin"] == label) & (income_grouped["race_short"] == race)]
        vals.append(row["interest_rate"].values[0] if not row.empty else np.nan)
    ax.bar(x + offsets[i], vals, width=width, label=race,
           color=COLORS[race], edgecolor="white", linewidth=0.5, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(income_labels, fontsize=11)
ax.set_xlabel("Annual Income Bracket", fontsize=12, labelpad=10)
ax.set_ylabel("Average Interest Rate (%)", fontsize=12, labelpad=10)
ax.set_title(
    "Average Interest Rate by Race — Controlled for Income\nNorth Carolina HMDA Data, 2019–2024",
    fontsize=14, fontweight="bold", pad=15
)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f%%"))
ax.legend(title="Race", fontsize=10, title_fontsize=10, framealpha=0.8)
ax.grid(axis="y", linestyle="--", alpha=0.5, zorder=0)
ax.spines[["top", "right"]].set_visible(False)
ax.annotate(
    "Within each income bracket, if rate gaps persist by race,\nthey cannot be attributed to income differences alone.",
    xy=(0.02, 0.97), xycoords="axes fraction", fontsize=9, color="#555555", va="top",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="#CCCCCC", alpha=0.8)
)
plt.tight_layout()
plt.savefig("charts/chart2_rate_by_income_and_race.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: charts/chart2_rate_by_income_and_race.png")

# ===========================================
# CHART 3 — DTI + Income Combined (double-controlled)
# ===========================================
df_mid = df[
    (df["dti_bin"].isin(["20-30%", "30-40%", "40-50%"])) &
    (df["income_bin"].isin(["$50-75k", "$75-100k", "$100-150k", "$150k+"]))
].copy()

df_mid["combined_bin"] = df_mid["income_bin"].astype(str) + " / DTI " + df_mid["dti_bin"].astype(str)

combined_grouped = (
    df_mid.groupby(["combined_bin", "race_short"], observed=True)["interest_rate"]
    .mean()
    .reset_index()
)

combined_labels = sorted(df_mid["combined_bin"].unique())
x = np.arange(len(combined_labels))

fig, ax = plt.subplots(figsize=(16, 6))
fig.patch.set_facecolor("#F7F9FB")
ax.set_facecolor("#F7F9FB")

for i, race in enumerate(races):
    vals = []
    for label in combined_labels:
        row = combined_grouped[(combined_grouped["combined_bin"] == label) & (combined_grouped["race_short"] == race)]
        vals.append(row["interest_rate"].values[0] if not row.empty else np.nan)
    ax.bar(x + offsets[i], vals, width=width, label=race,
           color=COLORS[race], edgecolor="white", linewidth=0.5, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(combined_labels, fontsize=8.5, rotation=20, ha="right")
ax.set_xlabel("Income Bracket / DTI Bracket", fontsize=12, labelpad=10)
ax.set_ylabel("Average Interest Rate (%)", fontsize=12, labelpad=10)
ax.set_title(
    "Average Interest Rate by Race — Controlled for Both Income and DTI\nNorth Carolina HMDA Data, 2019–2024",
    fontsize=14, fontweight="bold", pad=15
)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f%%"))
ax.legend(title="Race", fontsize=10, title_fontsize=10, framealpha=0.8)
ax.grid(axis="y", linestyle="--", alpha=0.5, zorder=0)
ax.spines[["top", "right"]].set_visible(False)
ax.annotate(
    "Borrowers in the same income AND DTI bracket — rate differences here\nare the hardest to explain through standard risk-based pricing.",
    xy=(0.02, 0.97), xycoords="axes fraction", fontsize=9, color="#555555", va="top",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="#CCCCCC", alpha=0.8)
)
plt.tight_layout()
plt.savefig("charts/chart3_rate_by_income_dti_race.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: charts/chart3_rate_by_income_dti_race.png")

print("\nAll 3 charts saved to charts/")

Saved: charts/chart1_rate_by_dti_and_race.png
Saved: charts/chart2_rate_by_income_and_race.png
Saved: charts/chart3_rate_by_income_dti_race.png

All 3 charts saved to charts/
